# 01 — TextRank (estrattivo)

**TextRank** (Mihalcea & Tarau, 2004) è un metodo estrattivo basato su grafi: le frasi del
documento sono i nodi, la similarità tra frasi pesa gli archi e un algoritmo tipo PageRank assegna
a ogni frase un punteggio di centralità. Il riassunto è composto dalle `N_SENTENCES` frasi con
punteggio più alto (nella loro forma originale). Nell'implementazione di **pyAutoSummarizer** la
similarità è la cosine similarity tra *sentence embeddings* (`all-MiniLM-L6-v2` di
sentence-transformers).

Questo notebook opera su due ambiti (`SCOPE`):
- **`sample`** — il campione condiviso prodotto da [00_prepara_campione.ipynb](00_prepara_campione.ipynb),
  per il confronto con i metodi abstractive (BART, PEGASUS);
- **`full`** — l'intero dataset pulito (`data/tab/complete.tab`, 56.101 esempi, letto in
  streaming), per il confronto tra i metodi estrattivi. ⚠️ Con gli embeddings su CPU la corsa
  completa richiede molte ore (~5,7 milioni di frasi da codificare): eseguirla sulla macchina con
  GPU NVIDIA (il notebook usa CUDA automaticamente se disponibile).

I riassunti generati vengono salvati incrementalmente in `results/summaries/textrank_{scope}.tsv`
(un'esecuzione interrotta **riprende** da dove era arrivata); le metriche vengono calcolate in una
sezione separata che legge solo i file salvati, quindi sono ricalcolabili senza rigenerare nulla.

In [1]:
# Installa le dipendenze se mancanti (per esempio su Google Colab)
try:
    import pyAutoSummarizer  # noqa: F401
except ImportError:
    %pip install pyAutoSummarizer sentencepiece

In [1]:
# --- Configurazione ---------------------------------------------------------
import summ_utils as su

METODO      = 'textrank'
SCOPE       = 'full'   # 'sample' = campione condiviso | 'full' = intero complete.tab
N_SAMPLES   = 100        # deve combaciare con il campione creato dal notebook 00
SEED        = 42
LIMIT       = None       # es. 3 per uno smoke test rapido; None = tutti
N_SENTENCES = 11         # frasi estratte per riassunto (mediana del corpus: 11 frasi/riassunto)
ST_MODEL    = 'all-MiniLM-L6-v2'  # modello di sentence embeddings
STOP_WORDS  = ['en']     # pre-processing per il ranking (l'output resta il testo originale)

BASE   = su.trova_base_dir()
P      = su.percorsi_standard(BASE)
SAMPLE_PATH = P['sample_dir'] / f'sample_{N_SAMPLES}_seed{SEED}.tsv'
OUT_PATH    = P['summaries_dir'] / f'{METODO}_{SCOPE}.tsv'
DEVICE = su.rileva_device()

print(f'Ambito (SCOPE) : {SCOPE}')
print(f'Device         : {DEVICE}')
print(f'Output         : {OUT_PATH}')

Ambito (SCOPE) : full
Device         : cuda
Output         : c:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\summaries\textrank_full.tsv


## Generazione dei riassunti

Per ogni documento: si sostituisce il separatore di articoli `` ||||| `` con un newline, si crea
un'istanza `psr.summarization` (il pre-processing — minuscole, rimozione stopword ecc. — serve solo
al *ranking*: le frasi restituite sono quelle originali), si calcola il rank con `summ_text_rank`
e si estraggono le prime `N_SENTENCES` frasi con `show_summary`.

Nota implementativa: la libreria carica il modello di embeddings **per ogni istanza** (cioè per
ogni documento). Per evitarlo, il modello viene caricato una sola volta qui e iniettato nella
cache `loaded_models` di ogni istanza — stesso identico comportamento, senza il costo di ricarica.

In [2]:
from pyAutoSummarizer.base import psr
from sentence_transformers import SentenceTransformer

# Modello di embeddings condiviso tra tutti i documenti (su GPU se disponibile)
modello_st = SentenceTransformer(ST_MODEL, device=DEVICE)

def genera(documento):
    s = psr.summarization(documento, stop_words=STOP_WORDS)
    s.loaded_models[ST_MODEL] = modello_st  # evita di ricaricare il modello per ogni documento
    rank = s.summ_text_rank(model=ST_MODEL)
    return s.show_summary(rank, n=N_SENTENCES)

esempi = (su.carica_campione(SAMPLE_PATH) if SCOPE == 'sample'
          else su.itera_complete_tab(P['complete_tab']))
scrittore = su.ScrittoreRiassunti(OUT_PATH)
errori = su.ciclo_summarization(esempi, scrittore, genera, limit=LIMIT, etichetta='TextRank ')
scrittore.chiudi()

c:\Users\antonio.girasella\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13125.12it/s]


TextRank [1] media 0.3 s/esempio (saltati 0 gia' fatti)
TextRank [2] media 0.2 s/esempio (saltati 0 gia' fatti)
TextRank [3] media 0.2 s/esempio (saltati 0 gia' fatti)
TextRank [10] media 0.1 s/esempio (saltati 0 gia' fatti)
TextRank [20] media 0.1 s/esempio (saltati 0 gia' fatti)
TextRank [30] media 0.0 s/esempio (saltati 0 gia' fatti)
TextRank [40] media 0.0 s/esempio (saltati 0 gia' fatti)
TextRank [50] media 0.0 s/esempio (saltati 0 gia' fatti)
TextRank [60] media 0.0 s/esempio (saltati 0 gia' fatti)
TextRank [70] media 0.0 s/esempio (saltati 0 gia' fatti)
TextRank [80] media 0.0 s/esempio (saltati 0 gia' fatti)
TextRank [90] media 0.0 s/esempio (saltati 0 gia' fatti)
TextRank [100] media 0.0 s/esempio (saltati 0 gia' fatti)
TextRank [110] media 0.0 s/esempio (saltati 0 gia' fatti)
TextRank [120] media 0.0 s/esempio (saltati 0 gia' fatti)
TextRank [130] media 0.0 s/esempio (saltati 0 gia' fatti)
TextRank [140] media 0.0 s/esempio (saltati 0 gia' fatti)
  ERRORE su row_id=143: Index

## Valutazione (indipendente dalla generazione)

Questa sezione legge **solo** i file salvati (`results/summaries/…` e il campione / complete.tab
per i riferimenti): può essere rieseguita in qualunque momento senza rigenerare i riassunti.
Metriche: ROUGE-1/2/L (F1, precisione, recall), BLEU e METEOR, calcolate con le funzioni di
pyAutoSummarizer su testo normalizzato in modo identico per tutti i metodi (minuscole, senza
punteggiatura, **con** stopword e numeri). Il riferimento è il riassunto umano, privato del
trattino editoriale iniziale «– ». Output: `results/metrics/textrank_{scope}_per_example.csv` e
`…_aggregate.json` (medie complessive e per split).

In [3]:
import json

riassunti   = su.carica_riassunti(OUT_PATH)
riferimenti = (su.carica_campione(SAMPLE_PATH) if SCOPE == 'sample'
               else su.itera_complete_tab(P['complete_tab']))
config = {'n_sentences': N_SENTENCES, 'st_model': ST_MODEL, 'stop_words': STOP_WORDS,
          'device': DEVICE, 'libreria': 'pyAutoSummarizer 1.2.0'}

righe, aggregato = su.valuta_e_salva(riferimenti, riassunti, METODO, SCOPE,
                                     P['metrics_dir'], config)
print(json.dumps(aggregato['overall'], indent=2))

Metriche per-esempio : c:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\textrank_full_per_example.csv (55894 righe)
Metriche aggregate   : c:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\textrank_full_aggregate.json
{
  "rouge1_f1": 0.36826566658308996,
  "rouge1_p": 0.32929795402103534,
  "rouge1_r": 0.44570556619889506,
  "rouge2_f1": 0.1348112856906645,
  "rouge2_p": 0.11523264655826643,
  "rouge2_r": 0.17691714107882017,
  "rougeL_f1": 0.17550329120104444,
  "rougeL_p": 0.1444983003669425,
  "rougeL_r": 0.245405986451188,
  "bleu": 0.08873642890980557,
  "meteor": 0.4755645885075611,
  "parole_generate": 367.3074748631338
}


## Ispezione qualitativa

Qualche coppia riassunto generato / riferimento umano, per farsi un'idea di cosa producono le
frasi estratte (troncate per leggibilità).

In [ ]:
riferimenti = (su.carica_campione(SAMPLE_PATH) if SCOPE == 'sample'
               else su.itera_complete_tab(P['complete_tab']))
su.mostra_esempi(riferimenti, riassunti, quanti=2)

--- row_id=425 (split=train) ---
GENERATO   : The Jewish campaign treasurer for a Republican candidate in Connecticut is defending a campaign mailer that depicts their Democratic Party opponent in what is being described as the “worst kind of anti-Semitism.” 
 
 “The full-color, double-sided mailer shows an altered image of Democrat Matt Lesser, who is Jewish, with large, beady eyes, holding fistfuls of hundred-dollar bills “We do know, though, the feelings that the flier is evoking — the juxtaposition of a Jewish candidate for office and m
RIFERIMENTO: Just days after the slaying of 11 Jewish congregants at a Pittsburgh synagogue, a GOP candidate for a state Senate seat in Connecticut is accused of sending a mailer using an "age-old anti-Semitic trope." The ad sent out by Ed Charamut includes what the Washington Post calls a "money-grubbing" picture (here) of smiling opponent Matt Lesser, clutching $100 bills with a "crazed look in his eyes." Lesser says the original image of him was 